# Section 1: Ingestion & Document Corpus Setup (20 Mins)

**Enterprise Use Case:** CloudCorp Smart Assistant (AskHR Agent)

**Notebook Goal:** Establish the Google Cloud environment, configure storage buckets, and ingest multi-section corporate policy documents into Google Cloud Storage (GCS).

**Prerequisites in GCP:**
1. Google Cloud Project with Billing Enabled
2. Enabled APIs: `aiplatform.googleapis.com` (Vertex AI), `storage.googleapis.com` (Cloud Storage)

> **Quick API Activation (Cloud Shell):** Run `gcloud services enable aiplatform.googleapis.com storage.googleapis.com` in Cloud Shell if not already enabled.


In [1]:
# Step 1.1: Install Required Google Cloud Client Libraries
!pip install -q google-cloud-aiplatform>=1.60.0 google-cloud-storage>=2.14.0 numpy


zsh:1: 1.60.0 not found


In [6]:
# Step: Authenticate & Configure GCP Project (Colab OAuth or Local Service Account Key from .env)
import os

# 1. Load environment constants dynamically from .env file if available
def load_env_file():
    for path in [".env", "../.env", "cloudcorp-llms-on-gcp-lab/.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        val = v.strip().strip('"').strip("'")
                        if val:
                            os.environ.setdefault(k.strip(), val)
            print(f"Loaded configuration from '{path}'")
            return

load_env_file()

# 2. Project Configuration & Quota Project Settings
PROJECT_ID = os.environ.get("PROJECT_ID", "").strip() or os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip() or "your-gcp-project-id"  # @param {type:"string"}
REGION = os.environ.get("REGION", "us-central1").strip()               # @param {type:"string"}
BUCKET_NAME = os.environ.get("BUCKET_NAME", f"{PROJECT_ID}-cloudcorp-policies").strip()

# Guardrail: Prevent running with unconfigured placeholder Project ID
if PROJECT_ID in ["your-gcp-project-id", "YOUR_PROJECT_ID", ""]:
    raise ValueError(
        "⚠️ ACTION REQUIRED: Please enter your Google Cloud Project ID in the PROJECT_ID field above! "
        "Ensure Vertex AI and Cloud Storage APIs are enabled."
    )

# Prevent 403 ADC Quota Project error (e.g. consumer projects/618104708054 in Cloud Shell)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID

# 3. Authentication Option A: Google Colab OAuth
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated via Google Colab OAuth.")
except ImportError:
    print("ℹ️ Running outside Google Colab (Cloud Shell / Local Machine / IDE).")

# 4. Authentication Option B: Service Account JSON Key via .env / Local File
sa_key_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "")
if sa_key_path:
    candidate_key_paths = [
        sa_key_path,
        os.path.join("..", sa_key_path),
        os.path.join("cloudcorp-llms-on-gcp-lab", sa_key_path),
        os.path.expanduser(f"~/Documents/{sa_key_path}"),
        os.path.expanduser(f"~/Downloads/{sa_key_path}")
    ]
    resolved_key = next((p for p in candidate_key_paths if os.path.isfile(p)), None)
    if resolved_key:
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.abspath(resolved_key)
        print(f"🔑 Loaded Service Account Key: {os.environ['GOOGLE_APPLICATION_CREDENTIALS']}")
    else:
        print(f"⚠️ Key file '{sa_key_path}' specified in .env not found on disk.")

# 5. Verify Active Credentials & Bind Quota Project
import google.auth
credentials = None
try:
    credentials, auth_project = google.auth.default(quota_project_id=PROJECT_ID)
    if hasattr(credentials, "with_quota_project"):
        credentials = credentials.with_quota_project(PROJECT_ID)
    account_id = getattr(credentials, "service_account_email", None) or getattr(credentials, "signer_email", "Google User Account (OAuth/ADC)")
    print(f"✅ Active Identity: {account_id}")
    print(f"✅ Quota Project bound: {PROJECT_ID}")
except Exception as e:
    print(f"⚠️ Warning: No active credentials detected ({e})")
    print("   • In Colab: Run auth.authenticate_user()")
    print("   • In Cloud Shell: Run 'gcloud auth application-default set-quota-project '")
    print("   • Locally: Set GOOGLE_APPLICATION_CREDENTIALS in .env or run 'gcloud auth application-default login'")

# 6. Initialize Vertex AI with explicit credentials
import vertexai
vertexai.init(project=PROJECT_ID, location=REGION, credentials=credentials)
print(f"✅ Vertex AI initialized successfully for project '{PROJECT_ID}' in '{REGION}'.")


Loaded configuration from '../.env'
ℹ️ Running outside Google Colab (Local Machine / IDE).
🔑 Loaded Service Account Key: /Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/gcp-key.json
✅ Active Identity: cloudcorp-lab-sa@upgradlabs-1751346914784.iam.gserviceaccount.com
✅ Vertex AI initialized successfully for project 'upgradlabs-1751346914784' in 'us-central1'.


In [7]:
import google.auth

try:
    credentials, project = google.auth.default()
    print("✅ Credentials Found!")
    print(f"Project ID: {project}")
    print(f"Account   : {getattr(credentials, 'service_account_email', 'User Account')}")
except Exception as e:
    print(f"❌ No GCP credentials found on this machine yet:\n{e}")

✅ Credentials Found!
Project ID: upgradlabs-1751346914784
Account   : cloudcorp-lab-sa@upgradlabs-1751346914784.iam.gserviceaccount.com


### Ingestion of Multi-Section Policy Corpus
Dynamic upload of `hr_pto_policy.txt`, `it_laptop_policy.txt`, and `remote_work_policy.txt` to Google Cloud Storage.

In [11]:
# Step 1.3: Upload Policy Documents from sample_policies/ to Google Cloud Storage (GCS)
import os
import glob
from google.cloud import storage

SAMPLE_POLICIES_DATA = {
    "hr_pto_policy.txt": '''# CloudCorp Global Human Resources Policy: Time Off, Leave of Absence, and Workplace Flexibility
Document ID: POL-HR-2026-v3
Last Revised: January 15, 2026
Applies to: All CloudCorp Global Employees (Full-time, Part-time, and Fixed-Contract)

---

### Section 1: Paid Time Off (PTO) Accrual & Eligibility
1.1 Full-Time Regular Employees accrue Paid Time Off (PTO) at a rate of 1.67 days per completed calendar month of service, totaling 20 days (160 hours) per full calendar year.
1.2 Part-Time Regular Employees scheduled for 20 to 31 hours per week accrue PTO on a pro-rata basis equivalent to 50% of the full-time allocation (10 days / 80 hours per year).
1.3 Temporary contractors, interns, and external vendors are not eligible for paid corporate PTO, but may be eligible for statutory unpaid personal days as required by local jurisdiction.
1.4 New hires begin accruing PTO on their official start date and may utilize accrued time off following the completion of their initial 30-day onboarding probationary period.

---

### Section 2: PTO Carry-Over, Expiration, and Separation Rules
2.1 Carry-Over Cap: Employees are permitted to carry over a maximum of 5 unused PTO days (40 hours) from the prior calendar year into the new calendar year.
2.2 Rollover Expiration Window: All carried-over PTO days must be scheduled and taken on or before March 31 (end of Q1) of the subsequent calendar year. Any carried-over hours remaining unused on April 1 are permanently forfeited and cannot be recovered or converted into cash compensation.
2.3 Payout upon Termination: CloudCorp does not pay out accrued but unused PTO upon voluntary resignation or involuntary termination, except in jurisdictions where statutory labor laws explicitly require mandatory encashment of earned vacation hours upon final settlement.

---

### Section 3: Request Procedures, Notice Periods, and Blackout Windows
3.1 Short-Duration Requests (1 to 2 consecutive days): Must be submitted via the CloudCorp HR Portal (Workday/CloudHR) at least 48 hours in advance to ensure team coverage.
3.2 Extended-Duration Requests (3 or more consecutive days): Must be submitted at least two (2) weeks (14 calendar days) in advance and require direct People Manager written approval.
3.3 Critical Business Blackout Windows: PTO requests spanning more than 2 consecutive working days during quarterly financial close periods (the last 5 business days of March, June, September, and December) and during CloudCorp Annual Global Summit (October 12–16) will not be approved, barring verified medical emergencies.
3.4 Overlapping Team Leave: To preserve business continuity, no more than 25% of any engineering, cloud operations, or customer support team may take PTO simultaneously. In the event of competing requests submitted on the same date, priority is granted based on date of submission.

---

### Section 4: Sick Leave, Mental Health, and Medical Certification
4.1 Annual Sick Leave Allocation: Full-time employees are provided 10 dedicated paid Sick and Wellness Days per calendar year, allocated upfront on January 1. Sick leave does not accrue month-to-month and does not carry over into the following year.
4.2 Permitted Usage: Sick days may be used for personal illness, medical appointments, mental health recovery, or caring for an immediate family member (spouse, child, parent).
4.3 Medical Certificate Requirement: If an employee is absent due to illness for more than three (3) consecutive business days, an official medical certificate signed by a licensed healthcare professional must be uploaded to CloudCorp Benefits within 48 hours of return to work.
4.4 Mental Wellness Days: Up to two (2) days of the annual sick leave allocation may be taken as preventative Mental Wellness Days with same-day morning notification (by 8:30 AM local time).

---

### Section 5: Parental, Caregiver, and Adoption Leave
5.1 Primary Caregiver Paid Parental Leave: CloudCorp provides up to sixteen (16) weeks of fully paid parental leave for full-time employees upon the birth, adoption, or foster placement of a child. This leave runs concurrently with statutory family leave laws.
5.2 Secondary Caregiver Paid Parental Leave: Secondary caregivers are entitled to six (6) weeks of fully paid leave, which may be taken in up to two distinct increments within the first 12 months following birth or placement.
5.3 Notice Requirement: Employees must inform HR and their manager at least thirty (30) days prior to the anticipated start date of parental leave.

---

### Section 6: Bereavement, Civic Duty, and Emergency Leave
6.1 Bereavement Leave: Employees are granted up to five (5) consecutive paid days for the loss of an immediate family member (spouse, child, parent, sibling) and up to three (3) paid days for extended family members (grandparent, in-laws).
6.2 Jury Duty and Civic Service: Up to two (2) weeks of paid leave is granted for mandatory jury service upon submission of the official court summons to HR.
6.3 Emergency Leave: Up to three (3) days of paid emergency leave may be authorized by HR in the event of natural disaster, severe property damage, or domestic emergency.

---

### Section 7: Unpaid Sabbatical and Extended Personal Leaves
7.1 Eligibility: Full-time employees with a minimum of three (3) continuous years of service at CloudCorp may apply for an unpaid sabbatical of between 30 and 90 calendar days for professional development, charitable initiatives, or personal pursuits.
7.2 Approvals: Unpaid sabbaticals require dual approval from the Department VP and the Chief People Officer (CPO) at least 60 days in advance.
7.3 Health Insurance: Group health, dental, and vision insurance coverage remains active during an approved sabbatical, provided the employee continues paying their standard employee payroll deductions.

---

### Section 8: Escalations & Policy Inquiries
8.1 For questions regarding policy interpretation, PTO balance discrepancies, or special accommodations, employees should contact HR Operations via Slack at `#ask-hr` or open an HR Support Ticket in ServiceNow under category `People Operations: Leave Request`.''',
    "it_laptop_policy.txt": '''# CloudCorp IT Hardware & Equipment Policy
Document ID: POL-IT-2026-v2
Last Revised: February 2, 2026
Applies to: All Global CloudCorp Personnel

---

### Section 1: Standard Hardware Refresh & Lifecycle
1.1 Standard corporate and engineering laptops are scheduled for proactive lifecycle refresh every thirty-six (36) months from initial provision date.
1.2 Requests for early hardware upgrades prior to the 36-month cycle require Department Director sign-off and cost-center allocation approval.

---

### Section 2: Damaged, Broken, or Malfunctioning Devices
2.1 When an employee experiences hardware damage (e.g., cracked or flickering display screens, broken chassis, liquid spills, or battery failure), the employee is eligible for an expedited device replacement.
2.2 An official IT incident ticket (with prefix INC-) must be generated detailing the issue summary, device symptoms, and urgency before equipment can be dispensed.
2.3 Screen fractures and motherboard failures are categorized as Priority-High (P2) hardware tickets and automatically assigned to the Hardware Fulfillment Desk with a target turnaround of 24 hours.

---

### Section 3: Loaner Devices and Pick-Up Protocols
3.1 Once a replacement ticket (INC-XXXX) is generated, the employee may immediately visit the IT Helpdesk (Floor 2, Tech Depot) during operating hours (8:00 AM – 6:00 PM) to claim an emergency loaner laptop.
3.2 Employees must quote their active incident ticket ID when collecting the loaner.
3.3 CloudCorp data backups from Google Drive and Cloud Workstations will be automatically synchronized to the replacement unit upon login.''',
    "remote_work_policy.txt": '''# CloudCorp Hybrid and Remote Work Policy
Document ID: POL-REMOTE-2026-v2
Last Revised: January 20, 2026
Applies to: All Global CloudCorp Personnel

---

### Section 1: Hybrid Work Eligibility and Schedule
1.1 Regular full-time employees are eligible for hybrid work, allowing remote work for up to three (3) business days per week, subject to direct manager approval.
1.2 Mandatory in-office collaboration days are designated on Tuesdays and Thursdays for all engineering, product, and leadership teams.

---

### Section 2: Core Working Hours and Availability
2.1 All employees working remotely must remain accessible via Slack and Google Meet during core team hours: 10:00 AM to 3:00 PM in their local office time zone.
2.2 Non-exempt (hourly) employees must track all remote working hours accurately in Workday and obtain advance authorization before working overtime.

---

### Section 3: Equipment, Internet, and Home Office Stipend
3.1 CloudCorp provides an annual tax-free Home Office Expense Stipend of $500 per calendar year to reimburse high-speed broadband, ergonomic chairs, monitors, and peripherals.
3.2 Reimbursement claims must be submitted via Concur within sixty (60) days of purchase accompanied by an itemized receipt.
3.3 CloudCorp IT maintains ownership of all issued computing hardware (laptops, monitors, yubikeys) regardless of remote work status.'''
}

def ensure_sample_policies():
    """Locates or auto-creates sample_policies/ so the notebook is 100% self-contained."""
    candidate_paths = ["sample_policies", "../sample_policies", "cloudcorp-llms-on-gcp-lab/sample_policies"]
    policies_dir = next((p for p in candidate_paths if os.path.isdir(p)), None)
    if policies_dir:
        return policies_dir

    # Auto-create in current environment (e.g., /content in Google Colab)
    policies_dir = "sample_policies"
    os.makedirs(policies_dir, exist_ok=True)
    for fname, text in SAMPLE_POLICIES_DATA.items():
        fpath = os.path.join(policies_dir, fname)
        if not os.path.exists(fpath):
            with open(fpath, "w", encoding="utf-8") as f:
                f.write(text.strip())
    print(f"Created fallback sample_policies directory with 3 enterprise policy files in '{policies_dir}'")
    return policies_dir

policies_dir = ensure_sample_policies()

print("1. Locating policy files...")
policies_dir = ensure_sample_policies()
policy_files = sorted(glob.glob(os.path.join(policies_dir, "*.txt")))
print(f"   Found {len(policy_files)} policy files in '{policies_dir}':", [os.path.basename(f) for f in policy_files])

print(f"\n2. Connecting to Google Cloud Storage (Project: '{PROJECT_ID}')...")
storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)

print(f"\n3. Ensuring bucket exists: gs://{BUCKET_NAME} in region '{REGION}'...")
try:
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION, timeout=15)
    print(f"   ✅ Created new GCS bucket: gs://{BUCKET_NAME}")
except Exception as e:
    print(f"   ℹ️ Using existing bucket or skipped creation: {e}")
    bucket = storage_client.bucket(BUCKET_NAME)

print("\n4. Uploading policy files to GCS...")
for file_path in policy_files:
    filename = os.path.basename(file_path)
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    blob = bucket.blob(filename)
    blob.upload_from_string(content, timeout=15)
    print(f"   ✅ Uploaded {filename} -> gs://{BUCKET_NAME}/{filename} ({len(content)} bytes)")

print("\n🎉 [SUCCESS] All files from sample_policies/ successfully uploaded to GCS!")


1. Locating policy files...
   Found 3 policy files in '../sample_policies': ['hr_pto_policy.txt', 'it_laptop_policy.txt', 'remote_work_policy.txt']

2. Connecting to Google Cloud Storage (Project: 'upgradlabs-1751346914784')...

3. Ensuring bucket exists: gs://upgradlabs-1751346914784-cloudcorp-policies in region 'us-central1'...
   ✅ Created new GCS bucket: gs://upgradlabs-1751346914784-cloudcorp-policies

4. Uploading policy files to GCS...
   ✅ Uploaded hr_pto_policy.txt -> gs://upgradlabs-1751346914784-cloudcorp-policies/hr_pto_policy.txt (6137 bytes)
   ✅ Uploaded it_laptop_policy.txt -> gs://upgradlabs-1751346914784-cloudcorp-policies/it_laptop_policy.txt (1625 bytes)
   ✅ Uploaded remote_work_policy.txt -> gs://upgradlabs-1751346914784-cloudcorp-policies/remote_work_policy.txt (1374 bytes)

🎉 [SUCCESS] All files from sample_policies/ successfully uploaded to GCS!


In [12]:
# Step 1.4: Verify Uploaded Blobs in GCS
print(f"Listing blobs in gs://{BUCKET_NAME}:")
for blob in bucket.list_blobs():
    print(f"  • Blob: {blob.name} | Size: {blob.size} bytes | Content Type: {blob.content_type}")


Listing blobs in gs://upgradlabs-1751346914784-cloudcorp-policies:
  • Blob: hr_pto_policy.txt | Size: 6139 bytes | Content Type: text/plain
  • Blob: it_laptop_policy.txt | Size: 1627 bytes | Content Type: text/plain
  • Blob: remote_work_policy.txt | Size: 1374 bytes | Content Type: text/plain
